<a href="https://colab.research.google.com/github/nadia2622/UTS_KecerdasanBuatan/blob/main/UTS_KecerdasaBuatan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Di sini kami menyiapkan parameter GA. Panjang kromosom kami buat 20 bit: 10 bit pertama untuk x1, 10 bit terakhir untuk x2. Dengan 10 bit, kami mendapat 2¹⁰ = 1024 kemungkinan nilai per variabel, cukup untuk presisi di domain [-10, 10]. Pc = 0.8 dan Pm = 0.01 adalah nilai standar di literatur GA — Pc tinggi agar eksplorasi lewat crossover sering terjadi, Pm rendah agar tidak merusak solusi bagus.

In [ ]:
# ==========================================================
# TUGAS: Case Based - Searching (Genetic Algorithm)
# Fungsi yang diminimumkan:
#   f(x1, x2) = -( sin(x1)*cos(x2)*tan(x1+x2) + 0.5*exp(1 - sqrt(x2^2)) )
# Domain: -10 <= x1, x2 <= 10
# ==========================================================

# Modul bawaan Python (bukan library GA). Hanya untuk bilangan acak & fungsi matematika.
import random
import math

# pake seed biar setiap kali nge shuffle, populasi awal akan selalu dimulai dengan kromosom yang sama
random.seed(42)

# ===== Parameter GA =====
POPULATION_SIZE   = 50      # banyaknya kromosom dalam satu populasi
GENE_LENGTH       = 10      # jumlah bit per variabel (x1 atau x2)
CHROMOSOME_LENGTH = 2 * GENE_LENGTH  # total bit per kromosom = 20 bit
X_MIN, X_MAX      = -10, 10 # domain x1 dan x2

CROSSOVER_RATE    = 0.8     # Pc: probabilitas terjadinya pindah silang
MUTATION_RATE     = 0.01    # Pm: probabilitas tiap bit termutasi
MAX_GENERATIONS   = 100     # kriteria penghentian evolusi

2. Ini persis seperti tabel 'Populasi Awal' di catatan kuliah yang berisi kromosom 0011, 0101, 1010, 1100. Bedanya kami pakai 20 bit dan 50 kromosom karena masalah kami lebih kompleks.

In [ ]:
def initialize_population():
    """
    TAHAP 1 (bagian Populasi Awal):
    Buat POPULATION_SIZE kromosom acak, masing-masing berisi 20 bit (0 atau 1).
    Ini seperti kolom 'Kromosom' di tabel Populasi Awal pada catatan kuliah.
    """
    population = []
    for _ in range(POPULATION_SIZE):
        # 1 kromosom = list 20 angka acak 0 atau 1
        chromosome = [random.randint(0, 1) for _ in range(CHROMOSOME_LENGTH)]
        population.append(chromosome)
    return population

3. Di catatan kuliah, kolom 'Desimal' langsung jadi input fungsi. Tapi di tugas kami domain-nya [-10, 10], bukan bilangan bulat. Jadi kami tambahkan langkah pemetaan linear: desimal 0 → x = -10, desimal 1023 → x = 10, dan nilai antaranya proporsional. Ini umum dipakai di GA encoding biner.

In [ ]:
def decode_chromosome(chromosome):
    """
    Ubah kromosom biner menjadi nilai x1 dan x2 dalam range [-10, 10].

    Langkah:
    1. Pisahkan kromosom: 10 bit pertama = gen x1, 10 bit terakhir = gen x2.
    2. Konversi biner ke desimal (seperti di kuliah: 1010 -> 10).
    3. Petakan (map) nilai desimal ke range [-10, 10] dengan rumus linear:
           x = X_MIN + (X_MAX - X_MIN) * (desimal / desimal_maks)
    """
    # 1) Pisahkan menjadi dua gen
    gene1 = chromosome[:GENE_LENGTH]
    gene2 = chromosome[GENE_LENGTH:]

    # 2) Konversi biner ke desimal SECARA MANUAL (tanpa int(..., 2))
    def binary_to_decimal(bits):
        decimal = 0
        for i, bit in enumerate(bits):
            # bit paling kiri = most significant bit
            decimal += bit * (2 ** (GENE_LENGTH - 1 - i))
        return decimal

    dec1 = binary_to_decimal(gene1)
    dec2 = binary_to_decimal(gene2)

    # 3) Petakan ke domain [-10, 10]
    max_decimal = (2 ** GENE_LENGTH) - 1   # 1023 untuk 10 bit
    x1 = X_MIN + (X_MAX - X_MIN) * (dec1 / max_decimal)
    x2 = X_MIN + (X_MAX - X_MIN) * (dec2 / max_decimal)

    return x1, x2

4. Di catatan kuliah, contohnya memakai f = x² + 2x yang selalu positif dan soalnya MAKSIMISASI, jadi fitness = nilai fungsi objektif langsung. Sedangkan tugas kami adalah MINIMISASI dan f bisa negatif. Kalau kami pakai Roulette Wheel mentah, probabilitas bisa negatif — itu tidak boleh. Solusinya: kami transformasikan fitness = 1 / (1 + f_digeser). Yang f-nya paling kecil dapat fitness tertinggi (= 1), dan semua fitness dijamin positif. Ini trik standar untuk adaptasi Roulette Wheel ke masalah minimisasi.

In [ ]:
def objective_function(x1, x2):
    """
    TAHAP 1 (Fungsi Objektif):
    f(x1, x2) = -( sin(x1)*cos(x2)*tan(x1+x2) + 0.5*exp(1 - sqrt(x2^2)) )
    Catatan: sqrt(x2^2) = |x2|
    """
    try:
        term1 = math.sin(x1) * math.cos(x2) * math.tan(x1 + x2)
        term2 = 0.5 * math.exp(1 - math.sqrt(x2 ** 2))
        f = -(term1 + term2)

        # Tangani NaN/Inf (tan bisa meledak dekat pi/2)
        if math.isnan(f) or math.isinf(f):
            return 1e10   # "dihukum" dengan nilai sangat besar
        return f
    except (OverflowError, ValueError):
        return 1e10

def calculate_fitness(chromosome, f_min_population):
    """
    TAHAP 1 (Nilai Fitness):
    Karena tujuan kita MINIMUMKAN f, sedangkan Roulette Wheel butuh fitness
    yang "semakin besar = semakin baik", kita transformasikan:

        fitness = 1 / (1 + (f - f_min_population))

    - Geser f supaya tidak negatif (f - f_min >= 0)
    - Lalu invert: f terkecil -> fitness tertinggi (= 1)
    - Fitness selalu > 0 -> aman untuk Roulette Wheel
    """
    x1, x2 = decode_chromosome(chromosome)
    f = objective_function(x1, x2)
    fitness = 1 / (1 + (f - f_min_population))
    return fitness, f

5. Ini persis seperti tabel di catatan kuliah kolom P[i] dan C. Contoh di catatan: untuk kromosom 0011 dengan fitness 15 dan total 338, P[i] = 15/338 = 0.0444, dan C-nya juga 0.0444 karena yang pertama. Kromosom kedua, C-nya = 0.0444 + 0.1036 = 0.1480. Yaaa kami ikutin cara itu intinya :>

In [ ]:
def compute_probabilities(fitnesses):
    """
    TAHAP 1 (P[i], C, Interval)

    Input : list fitness semua kromosom
    Output: (probabilities, cumulatives)
        - probabilities[i] = fitness[i] / total_fitness   <- P[i]
        - cumulatives[i]   = cumulatives[i-1] + P[i]      <- C
        Interval kromosom ke-i = [cumulatives[i-1], cumulatives[i]]
    """
    total_fitness = sum(fitnesses)

    # P[i] = fitness[i] / total_fitness
    probabilities = [f / total_fitness for f in fitnesses]

    # C[i] = C[i-1] + P[i]  (cumulative)
    cumulatives = []
    running_sum = 0
    for p in probabilities:
        running_sum += p
        cumulatives.append(running_sum)

    return probabilities, cumulatives

6. Di catatan kuliah, contohnya: jika r = 0.2, cek jatuh di interval mana — ternyata 0.1480 - 0.5030, maka parent yang terpilih adalah 1010. Kami implementasikan persis begitu. Logika r <= c otomatis menemukan interval yang mengandung r karena kita cek dari interval pertama secara berurutan.

In [ ]:
def roulette_wheel_selection(population, cumulatives):
    """
    TAHAP 2: Seleksi Roulette Wheel.

    Cara kerja:
    1. Bangkitkan angka acak r dalam [0, 1).
    2. Cari kromosom yang interval cumulative-nya mencakup r.
       Contoh di kuliah: r = 0.2 jatuh di interval 0.1480 - 0.5030 -> pilih 1010.
    """
    r = random.random()  # angka acak antara 0 dan 1

    # Cari kromosom pertama yang cumulative-nya >= r
    for i, c in enumerate(cumulatives):
        if r <= c:
            # Kembalikan SALINAN (pakai [:]) supaya aslinya tidak terubah
            return population[i][:]

    # Fallback (sangat jarang, hanya jika ada error pembulatan)
    return population[-1][:]

7. Di catatan kuliah, titik potongnya tetap di posisi ke-2. Kami pakai titik potong acak agar eksplorasi lebih baik. Prinsipnya sama: potong dua kromosom orangtua di satu titik, lalu tukar bagian belakangnya. Probabilitas Pc = 0.8 berarti 80% kemungkinan crossover terjadi; 20% anak identik dengan orangtua.

In [ ]:
def crossover(parent1, parent2):
    """
    TAHAP 3: Single-point Crossover.

    Persis catatan kuliah:
    - Tentukan titik potong (di kuliah: titik ke-2).
    - Kromosom sebelum titik potong TETAP, setelah titik potong DITUKAR.
    - Contoh kuliah:
        parent1 = 1010, parent2 = 1100, titik potong = 2
        child1  = 10|00  (depan parent1 + belakang parent2)
        child2  = 11|10  (depan parent2 + belakang parent1)
    - Crossover hanya terjadi dengan probabilitas Pc. Jika tidak,
      anak = salinan orangtua.
    """
    if random.random() < CROSSOVER_RATE:
        # Titik potong acak antara 1 s/d panjang-1
        point = random.randint(1, CHROMOSOME_LENGTH - 1)

        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2
    else:
        # Tidak terjadi crossover -> salin orangtua
        return parent1[:], parent2[:]

8. Kemaren kan pas kita belajar ini, kalau bit yang dipilih = 0, jadi 1; kalau 1 jadi 0. Bedanya, di kmrn tuh mutasi dilakukan secara manual di bit tertentu. Kami otomatiskan: setiap bit kami cek satu per satu, dengan probabilitas Pm = 0.01 (1%) untuk dibalik. Dengan 20 bit, rata-rata 0.2 bit yang termutasi per kromosom — cukup untuk menjaga keragaman tanpa merusak solusi bagus.

In [ ]:
def mutate(chromosome):
    """
    TAHAP 4: Bit-flip Mutation.

    Persis catatan kuliah:
    "mutasi adalah mengubah nilai pada bit ke-n dari 1->0 ataupun 0->1"

    Setiap bit punya peluang Pm (MUTATION_RATE) untuk dibalik.
    Contoh kuliah: anak 1110 dimutasi di bit ke-4 -> 1111.
    """
    for i in range(len(chromosome)):
        if random.random() < MUTATION_RATE:
            # Trik flip: 1-0=1, 1-1=0
            chromosome[i] = 1 - chromosome[i]
    return chromosome

9. Ini tempat semua tahap sebelumnya dirangkai. Satu pasang orangtua menghasilkan dua anak. Kami ulangi sampai jumlah populasi baru sama dengan populasi lama. Nahh kmrn kan kita belajar, tahap 5 adalah 'Evaluasi Populasi Baru'.... nah, evaluasi itu kami lakukan di loop utama (cell berikutnya) dengan cara menghitung ulang fitness untuk populasi yang baru ini.

In [ ]:
def create_next_generation(population, cumulatives):
    """
    TAHAP 5: Pergantian Generasi.

    Gabungan Tahap 2, 3, 4 untuk menghasilkan populasi baru:
    1. Pilih 2 orangtua via Roulette Wheel (Tahap 2).
    2. Lakukan Crossover (Tahap 3) -> dapat 2 anak.
    3. Lakukan Mutasi (Tahap 4) pada tiap anak.
    4. Ulangi sampai populasi baru penuh.

    Setelah populasi baru terbentuk, di loop utama akan dilakukan
    evaluasi fitness baru (kembali ke Tahap 1).
    """
    new_population = []

    while len(new_population) < POPULATION_SIZE:
        # Tahap 2: Pilih dua orangtua
        parent1 = roulette_wheel_selection(population, cumulatives)
        parent2 = roulette_wheel_selection(population, cumulatives)

        # Tahap 3: Crossover
        child1, child2 = crossover(parent1, parent2)

        # Tahap 4: Mutasi
        child1 = mutate(child1)
        child2 = mutate(child2)

        # Tambahkan anak ke populasi baru
        new_population.append(child1)
        if len(new_population) < POPULATION_SIZE:
            new_population.append(child2)

    return new_population

10. Ini kerangka utamanya. Satu iterasi for = satu generasi. Di setiap generasi kami lakukan Tahap 1 lengkap (objektif → fitness → P[i] → C), lalu Tahap 2–5 untuk membuat populasi baru. Solusi terbaik sepanjang evolusi disimpan terpisah agar tidak hilang — ini penting karena tanpa elitism, solusi bagus bisa hilang saat crossover atau mutasi. Kami berhenti setelah 100 generasi, yang biasanya sudah cukup untuk konvergen.

In [ ]:
def run_genetic_algorithm(verbose=True):
    """
    Loop utama Genetic Algorithm.
    Setiap generasi = 1 siklus lengkap Tahap 1 sampai Tahap 5.
    Berhenti setelah MAX_GENERATIONS generasi (kriteria penghentian).
    """
    # === Inisialisasi Populasi Awal (Tahap 1a) ===
    population = initialize_population()

    # Variabel untuk menyimpan solusi terbaik sepanjang evolusi
    best_chromosome_ever = None
    best_f_ever          = float('inf')  # cari nilai terkecil -> mulai dari besar

    if verbose:
        print("=" * 70)
        print("MULAI EVOLUSI GENETIC ALGORITHM")
        print("=" * 70)

    for generation in range(MAX_GENERATIONS):
        # =========================================================
        # TAHAP 1: Evaluasi Fungsi Objektif, Fitness, P[i], C
        # =========================================================
        # Hitung f terlebih dulu supaya dapat f_min untuk normalisasi fitness
        f_values = []
        for chrom in population:
            x1, x2 = decode_chromosome(chrom)
            f_values.append(objective_function(x1, x2))

        f_min_pop = min(f_values)

        # Hitung fitness (sudah dinormalisasi) untuk semua kromosom
        fitnesses = []
        for chrom in population:
            fit, _ = calculate_fitness(chrom, f_min_pop)
            fitnesses.append(fit)

        # Hitung P[i] dan Cumulative C
        probabilities, cumulatives = compute_probabilities(fitnesses) # cell 5

        # --- Catat kromosom terbaik di generasi ini ---
        # Terbaik = yang f-nya paling kecil (karena minimisasi)
        best_idx_gen = f_values.index(min(f_values))
        f_best_gen   = f_values[best_idx_gen]

        # Update record keseluruhan
        if f_best_gen < best_f_ever:
            best_f_ever          = f_best_gen
            best_chromosome_ever = population[best_idx_gen][:]

        # Tampilkan progres tiap 10 generasi
        if verbose and (generation % 10 == 0 or generation == MAX_GENERATIONS - 1):
            x1_best, x2_best = decode_chromosome(population[best_idx_gen])
            print(f"Generasi {generation:3d} | "
                  f"f terbaik = {f_best_gen:10.6f} | "
                  f"x1 = {x1_best:7.4f} | x2 = {x2_best:7.4f}")

        # =========================================================
        # TAHAP 2, 3, 4, 5: Buat populasi baru
        # (Roulette Wheel -> Crossover -> Mutasi -> Pergantian)
        # =========================================================
        population = create_next_generation(population, cumulatives)

    if verbose:
        print("=" * 70)
        print("EVOLUSI SELESAI")
        print("=" * 70)

    return best_chromosome_ever, best_f_ever

11. Output program menampilkan tiga hal yang diminta di soal: kromosom terbaik dalam bentuk biner (20 bit), hasil dekode-nya dalam x1 dan x2, dan nilai f-nya. Dengan ini dosen bisa langsung memverifikasi bahwa hasilnya masuk akal.

In [ ]:
# Jalankan GA
best_chromosome, best_f = run_genetic_algorithm(verbose=True)

# Dekode kromosom terbaik
x1_best, x2_best = decode_chromosome(best_chromosome)

# ================= Tampilkan output sesuai permintaan soal =================
print("\n" + "=" * 70)
print(" HASIL AKHIR GENETIC ALGORITHM ".center(70, "="))
print("=" * 70)

# Ubah list bit ke string biner supaya mudah dibaca
chrom_str = ''.join(str(bit) for bit in best_chromosome)
gene1_str = chrom_str[:GENE_LENGTH]
gene2_str = chrom_str[GENE_LENGTH:]

print(f"Kromosom terbaik (20 bit) : {chrom_str}")
print(f"  - Gen x1 (10 bit)       : {gene1_str}")
print(f"  - Gen x2 (10 bit)       : {gene2_str}")
print(f"Nilai x1                  : {x1_best:.6f}")
print(f"Nilai x2                  : {x2_best:.6f}")
print(f"Nilai f(x1, x2) minimum   : {best_f:.6f}")
print("=" * 70)

MULAI EVOLUSI GENETIC ALGORITHM
Generasi   0 | f terbaik =  -3.475140 | x1 = -7.3607 | x2 = -0.1271
Generasi  10 | f terbaik = -29.555160 | x1 = -5.8162 | x2 =  4.2522
Generasi  20 | f terbaik = -29.555160 | x1 = -5.8162 | x2 =  4.2522
Generasi  30 | f terbaik = -29.555160 | x1 = -5.8162 | x2 =  4.2522
Generasi  40 | f terbaik = -29.555160 | x1 = -5.8162 | x2 =  4.2522
Generasi  50 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
Generasi  60 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
Generasi  70 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
Generasi  80 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
Generasi  90 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
Generasi  99 | f terbaik = -31.899133 | x1 = -5.7967 | x2 =  4.2326
EVOLUSI SELESAI

=================== HASIL AKHIR GENETIC ALGORITHM ====================
Kromosom terbaik (20 bit) : 00110101111011011000
  - Gen x1 (10 bit)       : 0011010111
  - Gen x2 (10 bit)       : 1011011000
Nilai